# Project - Airline AI Assistant

We'll now bring together what we've learned to make an AI Customer Support assistant for an Airline

In [50]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [51]:
# Initialization


ollama_url = "http://localhost:11434/v1"
ollama = OpenAI(api_key="ollama", base_url=ollama_url)


MODEL = 'llama3.2'

# As an alternative, if you'd like to use Ollama instead of OpenAI
# Check that Ollama is running for you locally (see week1/day2 exercise) then uncomment these next 2 lines
# MODEL = "llama3.2"
# openai = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')


In [52]:
system_message = """
You are a travel assistant.

Rules:
- Use get_ticket_price when user asks for ticket prices.
- Use set_ticket_price when user wants to update or set a price.
- Extract city and price correctly.
- Do not guess values.
- Call tools only when necessary.
"""

In [53]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = ollama.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content

gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7870
* To create a public link, set `share=True` in `launch()`.


## Tools

Tools are an incredibly powerful feature provided by the frontier LLMs.

With tools, you can write a function, and have the LLM call that function as part of its response.

Sounds almost spooky.. we're giving it the power to run code on our machine?

Well, kinda.

In [54]:
VALID_CITIES = {"london", "paris", "tokyo", "berlin"}

def handle_tool_calls(message):
    responses = []

    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get("destination_city", "").lower()

            if city not in VALID_CITIES:
                content = f"Invalid city: {city}"
            else:
                content = get_ticket_price(city)

            responses.append({
                "role": "tool",
                "content": content,
                "tool_call_id": tool_call.id
            })

    return responses

In [55]:
get_ticket_price("London")

DATABASE TOOL CALLED: Getting price for London


'Ticket price to London is $799.0'

In [56]:
# There's a particular dictionary structure that's required to describe our function:

price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

set_price_function = {
    "name": "set_ticket_price",
    "description": "Set or update the price of a ticket for a specific city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city to update"
            },
            "price": {
                "type": "string",
                "description": "The ticket price (e.g. $999)"
            }
        },
        "required": ["destination_city", "price"],
        "additionalProperties": False
    }
}

In [57]:
tools = [
    {"type": "function", "function": price_function},
    {"type": "function", "function": set_price_function}
]

In [58]:
tools

[{'type': 'function',
  'function': {'name': 'get_ticket_price',
   'description': 'Get the price of a return ticket to the destination city.',
   'parameters': {'type': 'object',
    'properties': {'destination_city': {'type': 'string',
      'description': 'The city that the customer wants to travel to'}},
    'required': ['destination_city'],
    'additionalProperties': False}}},
 {'type': 'function',
  'function': {'name': 'set_ticket_price',
   'description': 'Set or update the price of a ticket for a specific city.',
   'parameters': {'type': 'object',
    'properties': {'destination_city': {'type': 'string',
      'description': 'The city to update'},
     'price': {'type': 'string',
      'description': 'The ticket price (e.g. $999)'}},
    'required': ['destination_city', 'price'],
    'additionalProperties': False}}}]

## Getting OpenAI to use our Tool

There's some fiddly stuff to allow OpenAI "to call our tool"

What we actually do is give the LLM the opportunity to inform us that it wants us to run the tool.

Here's how the new chat function looks:

In [59]:
def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]

    messages = (
        [{"role": "system", "content": system_message}]
        + history
        + [{"role": "user", "content": message}]
    )

    # 🔴 Only enable tools if message looks like pricing request
    use_tools = any(word in message.lower() for word in ["price", "ticket", "cost", "travel"])

    response = ollama.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=tools if use_tools else None
    )

    choice = response.choices[0]

    if hasattr(choice.message, "tool_calls") and choice.message.tool_calls:
        message = choice.message

        tool_responses = handle_tool_calls(message)

        if tool_responses:  # only proceed if valid tool call exists
            messages.append(message)
            messages.extend(tool_responses)

            response = ollama.chat.completions.create(
                model=MODEL,
                messages=messages
            )

            return response.choices[0].message.content

    return choice.message.content

In [60]:
def handle_tool_calls(message):
    responses = []

    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get("destination_city")

            price_details = get_ticket_price(city)

            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })

    return responses

In [61]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7871
* To create a public link, set `share=True` in `launch()`.


## Let's make a couple of improvements

Handling multiple tool calls in 1 response

Handling multiple tool calls 1 after another

In [62]:
def chat(user_message, history):
    # normalize history
    history = [{"role": h["role"], "content": h["content"]} for h in history]

    messages = (
        [{"role": "system", "content": system_message}]
        + history
        + [{"role": "user", "content": user_message}]
    )

    # ✅ Only enable tools when needed
    use_tools = any(word in user_message.lower() for word in ["price", "ticket", "cost", "travel"])

    response = ollama.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=tools if use_tools else None
    )

    choice = response.choices[0]

    # ✅ Handle tool calls safely
    if hasattr(choice.message, "tool_calls") and choice.message.tool_calls:
        tool_responses = handle_tool_calls(choice.message)

        # Only continue if we got valid tool output
        if tool_responses:
            messages.append(choice.message)
            messages.extend(tool_responses)

            response = ollama.chat.completions.create(
                model=MODEL,
                messages=messages
            )

            return response.choices[0].message.content

    # ✅ Default fallback
    return choice.message.content

In [63]:
VALID_CITIES = {"london", "paris", "tokyo", "berlin"}

def handle_tool_calls(message):
    responses = []

    for tool_call in message.tool_calls:
        name = tool_call.function.name

        try:
            arguments = json.loads(tool_call.function.arguments)
        except:
            continue

        # 🔍 GET PRICE
        if name == "get_ticket_price":
            city = arguments.get("destination_city", "").lower()
            content = get_ticket_price(city)

        # ✏️ SET PRICE
        elif name == "set_ticket_price":
            city = arguments.get("destination_city", "").lower()
            price = arguments.get("price")

            if not city or not price:
                continue

            set_ticket_price(city, price)
            content = f"Price for {city} updated to {price}"

        else:
            continue

        responses.append({
            "role": "tool",
            "content": content,
            "tool_call_id": tool_call.id
        })

    return responses

In [64]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7872
* To create a public link, set `share=True` in `launch()`.


In [65]:
def chat(user_message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]

    messages = (
        [{"role": "system", "content": system_message}]
        + history
        + [{"role": "user", "content": user_message}]
    )

    while True:
        response = ollama.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools
        )

        choice = response.choices[0]
        message = choice.message

        # ✅ If no tool calls → we are done
        if not hasattr(message, "tool_calls") or not message.tool_calls:
            return message.content

        # ✅ Handle tool calls
        tool_responses = handle_tool_calls(message)

        # If nothing valid came back → stop loop (prevents infinite loop)
        if not tool_responses:
            return message.content or "Sorry, I couldn't process that."

        messages.append(message)
        messages.extend(tool_responses)

In [66]:
import sqlite3


In [67]:
DB = "prices.db"

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute('CREATE TABLE IF NOT EXISTS prices (city TEXT PRIMARY KEY, price REAL)')
    conn.commit()

In [68]:
def get_ticket_price(city):
    print(f"DATABASE TOOL CALLED: Getting price for {city}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM prices WHERE city = ?', (city.lower(),))
        result = cursor.fetchone()
        return f"Ticket price to {city} is ${result[0]}" if result else "No price data available for this city"

In [69]:
get_ticket_price("London")

DATABASE TOOL CALLED: Getting price for London


'Ticket price to London is $799.0'

In [70]:
def set_ticket_price(city, price):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('INSERT INTO prices (city, price) VALUES (?, ?) ON CONFLICT(city) DO UPDATE SET price = ?', (city.lower(), price, price))
        conn.commit()

In [71]:
ticket_prices = {"london":799, "paris": 899, "tokyo": 1420, "sydney": 2999}
for city, price in ticket_prices.items():
    set_ticket_price(city, price)

In [72]:
get_ticket_price("Tokyo")

DATABASE TOOL CALLED: Getting price for Tokyo


'Ticket price to Tokyo is $1420.0'

In [73]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7873
* To create a public link, set `share=True` in `launch()`.


## Exercise

Add a tool to set the price of a ticket!

In [74]:
def handle_tool_calls(message):
    responses = []

    for tool_call in message.tool_calls:
        name = tool_call.function.name

        try:
            arguments = json.loads(tool_call.function.arguments)
        except:
            continue

        # 🔍 GET PRICE
        if name == "get_ticket_price":
            city = arguments.get("destination_city", "").lower()
            content = get_ticket_price(city)

        # ✏️ SET PRICE
        elif name == "set_ticket_price":
            city = arguments.get("destination_city", "").lower()
            price = arguments.get("price")

            if not city or not price:
                continue

            set_ticket_price(city, price)
            content = f"Price for {city} updated to {price}"

        else:
            continue

        responses.append({
            "role": "tool",
            "content": content,
            "tool_call_id": tool_call.id
        })

    return responses

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7874
* To create a public link, set `share=True` in `launch()`.


DATABASE TOOL CALLED: Getting price for 
DATABASE TOOL CALLED: Getting price for moscow


: 

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business Applications</h2>
            <span style="color:#181;">Hopefully this hardly needs to be stated! You now have the ability to give actions to your LLMs. This Airline Assistant can now do more than answer questions - it could interact with booking APIs to make bookings!</span>
        </td>
    </tr>
</table>